# AI-SPEAK — Inference

Run a trained blendshape model on any `.wav` file and get a CSV with 52 blendshape values per frame at 60 FPS.

**Input:** WAV audio file  
**Output:** CSV with columns = blendshape names, rows = frames at 60 FPS

Only audio features (MFCC or HuBERT) are used — no phoneme alignment needed.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL  = "https://github.com/YOUR_USERNAME/AI-SPEAK_catB.git"  # <-- set your repo URL
REPO_PATH = "/content/AI-SPEAK_catB"

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}

%cd {REPO_PATH}
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q librosa==0.10.1 soundfile==0.12.1 transformers==4.40.0 \
    pytorch-tcn gdown tqdm pandas scikit-learn matplotlib

## 2. Configuration

In [ ]:
# ============================================================
#  PATHS
# ============================================================
REPO_PATH = "/content/AI-SPEAK_catB"

# Path to the trained checkpoint on Drive
# Example: /content/drive/MyDrive/AI_SPEAK_results/gru_mfcc_20240601_120000/checkpoints/best_model.pt
CHECKPOINT = "/content/drive/MyDrive/AI_SPEAK_results/YOUR_SESSION/checkpoints/best_model.pt"

# Input WAV file (upload to /content/ or provide a Drive path)
WAV_FILE = "/content/test.wav"

# Output CSV path
OUTPUT_CSV = "/content/predictions.csv"

# Speaker ID used during training (spk08 or spk14)
SPEAKER = "spk08"

# HuBERT pre-computed features directory (only needed if model was trained with --audio hubert)
# Set to None to extract HuBERT on-the-fly (slower, downloads model ~360MB on first run)
HUBERT_DIR = None
# HUBERT_DIR = "/content/drive/MyDrive/AI_SPEAK_hubert_features"

# Sliding window parameters
CHUNK_SIZE = 120   # frames per window (120 = 2s at 60fps)
OVERLAP    = 60    # overlap between windows (60 = 50%)

print(f"Checkpoint: {CHECKPOINT}")
print(f"WAV file:   {WAV_FILE}")
print(f"Output:     {OUTPUT_CSV}")

## 3. (Optional) Upload a WAV File

Skip if your WAV file is already accessible at the path above.

In [ ]:
from google.colab import files

uploaded = files.upload()  # opens a file picker

# Update WAV_FILE to the uploaded file name
if uploaded:
    WAV_FILE = f"/content/{list(uploaded.keys())[0]}"
    print(f"Uploaded: {WAV_FILE}")

## 4. Run Inference

In [ ]:
import os
os.chdir(REPO_PATH)

hubert_flag = f"--hubert_dir {HUBERT_DIR}" if HUBERT_DIR else ""

!python scripts/infer.py \
    --checkpoint {CHECKPOINT} \
    --wav        {WAV_FILE} \
    --output     {OUTPUT_CSV} \
    --speaker    {SPEAKER} \
    --chunk_size {CHUNK_SIZE} \
    --overlap    {OVERLAP} \
    --device     cuda \
    {hubert_flag}

## 5. Inspect Output

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
print(f"Shape: {df.shape}  ({df.shape[0]} frames, {df.shape[1]} blendshapes)")
print(f"Duration: {df.shape[0] / 60:.2f}s at 60 FPS")
df.head()

## 6. Visualize Blendshapes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(OUTPUT_CSV)
fps = 60
time = [i / fps for i in range(len(df))]

# ---- Mouth blendshapes (most relevant for speech) ----
MOUTH_COLS = [
    "jawOpen", "mouthOpen", "mouthSmileLeft", "mouthSmileRight",
    "mouthFunnel", "mouthPucker", "mouthLeft", "mouthRight",
]
mouth_cols = [c for c in MOUTH_COLS if c in df.columns]

fig, axes = plt.subplots(len(mouth_cols), 1, figsize=(14, 2 * len(mouth_cols)), sharex=True)
if len(mouth_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, mouth_cols):
    ax.plot(time, df[col], linewidth=0.8)
    ax.set_ylabel(col, fontsize=8)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Predicted Mouth Blendshapes", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Heatmap of all 52 blendshapes over time ----
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv(OUTPUT_CSV)

fig, ax = plt.subplots(figsize=(16, 8))
im = ax.imshow(
    df.values.T,
    aspect="auto",
    vmin=0, vmax=1,
    cmap="viridis",
    interpolation="nearest",
)
ax.set_yticks(range(len(df.columns)))
ax.set_yticklabels(df.columns, fontsize=7)

n_ticks = 10
tick_frames = np.linspace(0, len(df) - 1, n_ticks, dtype=int)
ax.set_xticks(tick_frames)
ax.set_xticklabels([f"{f/60:.1f}s" for f in tick_frames])

ax.set_xlabel("Time")
ax.set_title("All 52 Blendshapes — Predicted")
plt.colorbar(im, ax=ax, label="Blendshape value (0-1)")
plt.tight_layout()
plt.show()

## 7. Download Results

In [ ]:
from google.colab import files
files.download(OUTPUT_CSV)